# Exploration & Brand Selection Notebook
### Kaggle Customer Support on Twitter Dataset (`thoughtvector/customer-support-on-twitter`)

This notebook covers:
1. Dataset loading and preliminary inspection.
2. Empirical brand comparison across support volume, reply rates, and thread length.
3. Conversation thread reconstruction.
4. Data-driven intent clustering and discovery.
5. Retrieval and pipeline inspection.

In [1]:
import os
import sys
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure project root is in path
sys.path.append('..')

RAW_CSV = '../data/raw/twcs_sample.csv'
df_raw = pd.read_csv(RAW_CSV, low_memory=False, on_bad_lines='skip')
print('Dataset shape:', df_raw.shape)
df_raw.head()

## 1. Brand Distribution & Comparison
We compare the top brands in customer support volume to determine which brand provides the most suitable foundation.

In [2]:
with open('../data/processed/brand_comparison.json', 'r') as f:
    brand_comp = json.load(f)

df_brands = pd.DataFrame.from_dict(brand_comp, orient='index')
df_brands[['customer_tweets', 'support_replies', 'approx_conversations', 'avg_conversation_length', 'percentage_with_brand_reply']]

## 2. Conversation Reconstructions
Inspect the reconstructed customer-brand interaction units.

In [3]:
with open('../data/processed/conversations.json', 'r') as f:
    convs = json.load(f)

print(f'Total reconstructed AmazonHelp conversations: {len(convs)}')
for c in convs[:3]:
    print('---')
    print('Customer:', c['customer_text'])
    print('Brand Reply:', c['brand_reply'])

## 3. Discovered Intent Taxonomy
Review the 12 discovered intent clusters and positive examples.

In [4]:
with open('../data/processed/intents.json', 'r') as f:
    intents = json.load(f)

for name, info in intents.items():
    print(f"Intent: {name}")
    print(f"  Desc: {info['description'][:90]}...")
    print(f"  Escalation Guidance: {info['escalation_guidance']}")
    print()

## 4. End-to-End Pipeline Demonstration

In [5]:
from src.pipeline import CustomerSupportPipeline
pipeline = CustomerSupportPipeline()

query = 'Where is my order? Tracking says delivered but nothing arrived.'
res = pipeline.process(query)
print('Intent:', res.predicted_intent, f'({res.intent_confidence:.2f})')
print('Decision:', res.decision)
print('Reason:', res.escalation_reason)
print('Draft Reply:', res.draft_reply)